In [94]:
import pandas as pd
import re

In [95]:
df = pd.read_json('/home/outis/TechBoy/Japan_Car_Import_Advisory_Platform/data/unified/all_cars_data.json')
# df.set_index("car_url", inplace=True)
df.sample()

,source,car_url,title,model_code,ref_no,registration_year,manufacture_year,grade,transmission,mileage,...,body_type,freight_amount,inspection_amount,insurance_amount,vehicle_price_breakdown,make,model,model_name,features,image_urls
94,sbtjapan,https://www.sbtjapan.com/used-cars/AK6351,2009/8 TOYOTA VOXY ZS,DBA-ZRR70W,AK6351,,2009,,AT,"134,000km",...,Wagon,1650,250,37,2410,,,,"[Air Conditioner, Audio Player, Left Side Powe...",[https://img.sbtjapan.com/img/carphoto/AK6351/...


In [96]:
identity_columns = ["source", "ref_no", "car_url"]
vehicle_columns = [
    "make",
    "model",
    "model_code",
    "grade",
    "manufacture_year",
    "mileage",
    "engine_capacity",
    "fuel_type",
    "transmission",
    "body_type",
    "drive_type",
    "seats",
    "doors",
    "steering",
    "exterior_color"
]
pricing_columns = [
    "currency",
    "vehicle_price_usd",
    "total_price_usd",
    "discount_rate"
]
logistics_columns = [
    "delivery_port",
    "location_japan"
]
import_cost_columns = [
    "shipping_cost_usd",
    "marine_insurance_usd",
    "cif_value_usd",
    "exchange_rate",
    "customs_value_kes",
    "import_duty_kes",
    "excise_duty_kes",
    "vat_kes",
    "idf_kes",
    "rdl_kes",
    "port_charges_kes",
    "clearing_fees_kes",
    "registration_cost_kes",
    "total_landed_cost_kes"
]
comparison_columns = [
    "local_market_price_kes",
    "price_difference_kes",
    "price_difference_pct",
    "import_vs_local_recommendation"
]
ml_feature_columns = [
    "vehicle_age",
    "mileage_per_year",
    "engine_size_liters",
    "source_platform"
]
prediction_columns = [
    "predicted_price_usd",
    "predicted_price_kes",
    "prediction_timestamp",
    "model_version"
]
asset_columns = [
    "features",
    "primary_image_url",
    "image_urls"
]

raw_archive_columns = [
    "title",
    "registration_year",
    "chassis_no",
    "vehicle_price_breakdown",
    "inspection_amount",
    "insurance_amount",
    "freight_amount",
    "model_name"
]

### Identity Columns Inspection

In [97]:
def inspect_df(df: pd.DataFrame, n: int = 5, sample_cols: list = None) -> None:
    """
    Print a comprehensive inspection report of a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
    n : int, default 5
        Number of rows to show with head/tail.
    sample_cols : list, optional
        Specific columns to inspect for missing values and dtypes.
        If None, all columns are used.
    """
    cols = sample_cols if sample_cols else df.columns
    df_subset = df[cols]
    
    sep = "=" * 70
    print(sep)
    print("DATAFRAME INSPECTION REPORT")
    print(sep)
    
    # Shape & size
    print(f"\n🔹 Shape (rows, cols): {df_subset.shape}")
    print(f"🔹 Size (total cells): {df_subset.size:,}")
    
    # Column list with dtypes
    print("\n🔹 Columns & dtypes:")
    for col in df_subset.columns:
        print(f"    {col:<30} {str(df_subset[col].dtype):>10}")
    
    # Memory usage
    mem = df_subset.memory_usage(deep=True).sum()
    print(f"\n🔹 Memory usage: {mem / (1024**2):.2f} MB")
    
    # Descriptive statistics (numeric columns only)
    num_cols = df_subset.select_dtypes(include='number').columns
    if len(num_cols) > 0:
        print(f"\n🔹 Numeric columns description (first {len(num_cols)} numeric columns):")
        print(df_subset[num_cols].describe().to_string())
    else:
        print("\n🔹 No numeric columns to describe.")
    
    # Random 
    print(f"\n🔹 Random {n} rows (sample):")
    print(df_subset.sample(n).to_string())
    
    print("\n" + sep)
    print("Inspection complete.")
    print(sep)

In [98]:
inspect_df(df, sample_cols=identity_columns)

DATAFRAME INSPECTION REPORT

🔹 Shape (rows, cols): (100, 3)
🔹 Size (total cells): 300

🔹 Columns & dtypes:
    source                                str
    ref_no                                str
    car_url                               str

🔹 Memory usage: 0.02 MB

🔹 No numeric columns to describe.

🔹 Random 5 rows (sample):
       source    ref_no                                                       car_url
68   sbtjapan    AK9100                     https://www.sbtjapan.com/used-cars/AK9100
55   sbtjapan    AK6516                     https://www.sbtjapan.com/used-cars/AK6516
90   sbtjapan    AJ9078                     https://www.sbtjapan.com/used-cars/AJ9078
63   sbtjapan    AK9054                     https://www.sbtjapan.com/used-cars/AK9054
17  beforward  CC553710  https://www.beforward.jp/nissan/laurel/cc553710/id/14573702/

Inspection complete.


### Vehicle Column Inspection

In [99]:
import pandas as pd
import re

# ------------------ Clean & prepare lookup lists ------------------
# Your provided makes (deduplicated, lowercased)
makes_list = [
    "Toyota", "Honda", "Nissan", "Mazda", "Suzuki", "Mitsubishi", "Daihatsu",
    "Subaru", "Hino", "Volkswagen", "Bmw", "Isuzu", "Lexus", "Mercedes", "Audi",
    "Volvo", "Land Rover", "Ford", "Peugeot", "Jeep", "Citroen", "Jaguar",
    "Hyundai", "Kia", "Nissan Diesel", "Komatsu", "Hitachi", "Yanmar",
    "Kubota", "Iseki", "Kawasaki", "Mitsuoka", "Sumitomo", "Denyo", "IHI",
    "Mz Speed", "Kato", "Kobelco", "Tadano", "TCM", "Sakai", "Yamaha",
    "Morooka", "Alfaromeo", "Abarth", "Aimgain", "AMG", "Austin Rover",
    "Aprilia", "Bentley", "Birkin", "Buick", "Bridgestone", "Buggati",
    "Bomag", "Cadillac", "Caterpillar", "Chevrolet", "Chrysler", "DAF",
    "Dodge", "Datsun", "Daimler", "Ducati", "Daewoo", "Ferrari", "Fiat",
    "Furukawa Unic", "Foden", "Geely", "GM", "Hummer", "Hanix",
    "Harley Davidson", "Iwafuji", "John Deere", "JCB", "KIA",
    "Kobashi Kogyo", "Lancia", "Lincoln", "Lamborghini", "MF", "Michelin",
    "Meiwa", "Mercury", "Maybach", "Mitsui Seiki", "MG Motor", "Maserati",
    "Mini", "NipponSharyo", "Nichiyu", "Opel", "Porsche", "Renault",
    "Rolls Royce", "Scania", "Ssangyong", "Skoda", "Smart", "Saab",
    "Sky Jack", "TOA", "Toyo Tyres", "Tesla", "TVR", "Tata Motors",
    "Triumph", "Vauxhall", "Yokohama"
]
makes = sorted(set(m.lower() for m in makes_list), key=lambda x: -len(x))  # longest first

# Your provided body types (deduplicated, lowercased, sorted by length desc)
body_types_raw = [
    "Sedan", "Coupe", "Hatchback", "Station Wagon", "SUV", "Pick up", "Van",
    "Mini Van", "Wagon", "Convertible", "Bus", "Truck", "Heavy Equipment",
    "Sedan Cars", "Jeep", "MUV", "Coupes", "Sports Cars", "Pickup Trucks",
    "All Vans", "Long Van", "Small Van", "All Buses", "Large Bus", "Mini Bus",
    "All Trucks", "Mini Trucks", "Dump Truck", "Flatbody Truck",
    "Box Body Truck", "Freezer Truck", "Crane Truck", "Wingbody Truck",
    "Tanker Truck", "Trailer Head", "Double Cabin", "Garbage Truck",
    "Vaccum Truck", "Concrete Pump", "Self Loader", "Car Carrier",
    "Fire Fighting", "Mixer Truck", "Trailer", "Cargo Truck", "Cabover",
    "Drilling Truck", "All Machinery", "Excavator", "Wheel Loader",
    "Bulldozer", "Roller", "Crawler", "Heavy Cranes", "Aerial Platform",
    "Generator", "Motor Grader", "Air Compressor", "Stone Crusher",
    "Asphalt Finisher", "Forklifts", "Farm Machinery", "Tractors",
    "Other Machinery", "Bikes", "Boats", "Parts", "Camper Van", "Bicycle"
]
body_types = sorted(set(bt.lower() for bt in body_types_raw), key=lambda x: -len(x))

# ------------------------------------------------------------------
def extract_from_title(title: str):
    """Return (make, model_name, model, body_type) from a title string."""
    if pd.isna(title) or not title.strip():
        return '', '', '', ''
    
    t = title.strip()
    found_make = ''
    found_body = ''
    
    # 1. Find make (longest match, case‑insensitive)
    for m in makes:
        # match whole word (or with delimiters) – we use re.search with word boundaries
        if re.search(r'\b' + re.escape(m) + r'\b', t, re.IGNORECASE):
            found_make = m.title()  # store nicely capitalized
            # remove that exact substring once (keep original case)
            t = re.sub(r'\b' + re.escape(m) + r'\b', '', t, count=1, flags=re.IGNORECASE).strip()
            break
    
    # 2. Find body type in the remaining string
    for bt in body_types:
        if re.search(r'\b' + re.escape(bt) + r'\b', t, re.IGNORECASE):
            found_body = bt.title()  # store original case
            # split around the matched body type
            pattern = re.compile(r'\b' + re.escape(bt) + r'\b', re.IGNORECASE)
            parts = pattern.split(t, maxsplit=1)
            before = parts[0].strip() if len(parts) > 0 else ''
            after = parts[1].strip() if len(parts) > 1 else ''
            t = (before + ' ' + after).strip()  # remove the body type
            # we will handle splitting using original split, not t
            # so store before/after directly:
            left = before
            right = after
            break
    else:
        # No body type found → entire remaining is “model_info”
        left = t
        right = ''
    
    # 3. Extract model_name (first token(s)) and model (rest)
    left_tokens = left.split()
    right_tokens = right.split()
    
    # Heuristic: model_name = first token (or first two if second is a number?)
    # For simplicity: model_name = first token, model = rest joined
    # But for "TOYOTA HIACE VAN DX", left = "HIACE", right = "DX"  → model_name="HIACE", model="DX"
    # For "HARRIER PREMIUM" with no body → left = "HARRIER PREMIUM", right = ""
    #   We want model_name="HARRIER", model="PREMIUM" → take left_tokens[0] and rest as model
    #   If right is empty, use left tokens 1.. as model.
    if left_tokens:
        model_name = left_tokens[0]
        # rest of left plus all right
        model_parts = left_tokens[1:] + right_tokens
        model = ' '.join(model_parts)
    else:
        model_name = ''
        model = ' '.join(right_tokens)
    
    # Clean up any stray digits/year from model_name/model if desired
    # Not strictly required but could remove leading year. We'll let it be.
    return found_make, model_name.strip(), model.strip(), found_body

# Apply to DataFrame
df[['make', 'model_name', 'model', 'body_type_from_title']] = df['title'].apply(
    lambda t: pd.Series(extract_from_title(t))
)

# Quick inspection
df[['title', 'make', 'model_name', 'model', 'body_type_from_title']].head(10)

,title,make,model_name,model,body_type_from_title
0,2009 TOYOTA VOXY ZS KIRAMEKI,Toyota,2009,VOXY ZS KIRAMEKI,
1,2020 TOYOTA HIACE VAN DX,Toyota,2020,HIACE DX,Van
2,1997 ISUZU ELF TRUCK,Isuzu,1997,ELF,Truck
3,2014 TOYOTA HARRIER PREMIUM,Toyota,2014,HARRIER PREMIUM,
4,2018 MERCEDES-BENZ C-CLASS C220D LAUREUS EDITION,Mercedes,2018,-BENZ C-CLASS C220D LAUREUS EDITION,
5,2017 NISSAN AD VAN NV150 VE EMERGENCY BRAKE,Nissan,2017,AD NV150 VE EMERGENCY BRAKE,Van
6,2012 MAZDA DEMIO 13C-V SMART EDITION 2,Smart,2012,MAZDA DEMIO 13C-V EDITION 2,
7,2015 NISSAN NOTE X DIG-S V SELECTION + SAFETY,Nissan,2015,NOTE X DIG-S V SELECTION + SAFETY,
8,2025 ISUZU D-MAX 2.2 X-RIDER DOUBLE CAB,Isuzu,2025,D-MAX 2.2 X-RIDER DOUBLE CAB,
9,2026 ISUZU D-MAX 2.2 X-RIDER DOUBLE CAB,Isuzu,2026,D-MAX 2.2 X-RIDER DOUBLE CAB,


In [100]:
inspect_df(df, sample_cols=vehicle_columns)

DATAFRAME INSPECTION REPORT

🔹 Shape (rows, cols): (100, 15)
🔹 Size (total cells): 1,500

🔹 Columns & dtypes:
    make                                  str
    model                                 str
    model_code                            str
    grade                                 str
    manufacture_year                      str
    mileage                               str
    engine_capacity                       str
    fuel_type                             str
    transmission                          str
    body_type                             str
    drive_type                            str
    seats                                 str
    doors                               int64
    steering                              str
    exterior_color                        str

🔹 Memory usage: 0.07 MB

🔹 Numeric columns description (first 1 numeric columns):
            doors
count  100.000000
mean     4.690000
std      0.677115
min      2.000000
25%      5.000000
50%      

In [101]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Missing report function (treats empty strings as missing)
# ------------------------------------------------------------------
def missing_report(df, columns, treat_empty_as_null=True):
    existing = [c for c in columns if c in df.columns]
    if not existing:
        return pd.DataFrame(columns=["missing_count", "missing_pct"])
    missing_counts = {}
    for col in existing:
        null_mask = df[col].isna()
        if treat_empty_as_null and df[col].dtype == object:
            safe = df[col].fillna('__PLACEHOLDER__')
            empty_mask = safe.astype(str).str.strip() == ''
        else:
            empty_mask = False
        missing_counts[col] = (null_mask | empty_mask).sum()
    missing = pd.Series(missing_counts)
    missing_pct = ((missing / len(df)) * 100).round(2)
    return pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

# ------------------------------------------------------------------
# 2. Report missing for all columns
# ------------------------------------------------------------------
all_cols = df.columns.tolist()
report = missing_report(df, all_cols)
print("=== Missing Value Report (empty strings counted as missing) ===")
print(report)

# ------------------------------------------------------------------
# 3. Define cleaning strategies
# ------------------------------------------------------------------
# These rules are examples – adjust them to your own logic.
clean_rules = {
    # Core identifiers → drop row if missing
    "car_url": "drop_row",
    "source": "drop_row",
    "title": "drop_row",
    "ref_no": "drop_row",

    # Text / categorical → fill with "Unknown"
    "make": "fill_unknown",
    "model_name": "fill_unknown",
    "model": "fill_unknown",
    "body_type": "fill_unknown",
    "grade": "fill_unknown",
    "transmission": "fill_unknown",
    "fuel_type": "fill_unknown",
    "drive_type": "fill_unknown",
    "steering": "fill_unknown",
    "exterior_color": "fill_unknown",
    "chassis_no": "fill_unknown",
    "delivery_port": "fill_unknown",
    "location": "fill_unknown",
    "discount_rate": "fill_unknown",

    # Numeric → fill with 0 (or median)
    "vehicle_price": "fill_0",
    "total_price": "fill_0",
    "freight_amount": "fill_0",
    "inspection_amount": "fill_0",
    "insurance_amount": "fill_0",
    "vehicle_price_breakdown": "fill_0",
    "original_price": "fill_0",
    "mileage_km": "fill_median",
    "engine_capacity_cc": "fill_median",
    "seats": "fill_0",
    "doors": "fill_0",
    "manufacturing_year": "fill_0",

    # Lists → fill with empty list
    "features": "fill_empty_list",
    "image_urls": "fill_empty_list",
}

# ------------------------------------------------------------------
# 4. Apply cleaning
# ------------------------------------------------------------------
def clean_missing(df, rules):
    df = df.copy()
    for col, action in rules.items():
        if col not in df.columns:
            continue

        # Identify missing values (including empty strings)
        null_mask = df[col].isna()
        if df[col].dtype == object:
            safe = df[col].fillna('__PLACEHOLDER__')
            empty_mask = safe.astype(str).str.strip() == ''
        else:
            empty_mask = False
        miss_mask = null_mask | empty_mask

        if miss_mask.sum() == 0:
            continue

        if action == "drop_row":
            df = df[~miss_mask]
        elif action == "fill_unknown":
            df[col] = df[col].mask(miss_mask, "Unknown")
        elif action == "fill_0":
            df[col] = df[col].mask(miss_mask, 0)
        elif action == "fill_median":
            median_val = df[col].median()
            df[col] = df[col].mask(miss_mask, median_val)
        elif action == "fill_empty_list":
            # Only for columns expected to be lists; if they are strings they will be set to []
            df[col] = df[col].mask(miss_mask, [])
        # else: leave as is
    return df

df_clean = clean_missing(df, clean_rules)

# ------------------------------------------------------------------
# 5. Verify result
# ------------------------------------------------------------------
report_after = missing_report(df_clean, all_cols)
print("\n=== After cleaning ===")
print(report_after)
print(f"Rows before: {len(df)}, after: {len(df_clean)}")
df_clean.head(3)

=== Missing Value Report (empty strings counted as missing) ===
                         missing_count  missing_pct
source                               0          0.0
car_url                              0          0.0
title                                0          0.0
model_code                           0          0.0
ref_no                               0          0.0
registration_year                    0          0.0
manufacture_year                     0          0.0
grade                                0          0.0
transmission                         0          0.0
mileage                              0          0.0
engine_capacity                      0          0.0
fuel_type                            0          0.0
seats                                0          0.0
doors                                0          0.0
steering                             0          0.0
drive_type                           0          0.0
exterior_color                       0          0.0


,source,car_url,title,model_code,ref_no,registration_year,manufacture_year,grade,transmission,mileage,...,freight_amount,inspection_amount,insurance_amount,vehicle_price_breakdown,make,model,model_name,features,image_urls,body_type_from_title
0,beforward,https://www.beforward.jp/toyota/voxy/cc101493/...,2009 TOYOTA VOXY ZS KIRAMEKI,DBA-ZRR75W,CC101493,2009/4,2009,ZS KIRAMEKI,Automatic,"178,375 km",...,0,0,0,0,Toyota,VOXY ZS KIRAMEKI,2009,"[CD Player, Power Steering, Power Window, A/C,...",[https://image-cdn.beforward.jp/large/202604/1...,
1,beforward,https://www.beforward.jp/toyota/hiace-van/cc88...,2020 TOYOTA HIACE VAN DX,QDF-GDH206V,CC889261,2020/3,-,DX,Automatic,"74,375 km",...,0,0,0,0,Toyota,HIACE DX,2020,"[Power Steering, Power Window, A/C, ABS, Airba...",[https://image-cdn.beforward.jp/large/202604/1...,Van
2,beforward,https://www.beforward.jp/isuzu/elf-truck/cd066...,1997 ISUZU ELF TRUCK,KC-NKR66ED,CD066063,1997/9,-,,Manual,"116,333 km",...,0,0,0,0,Isuzu,ELF,1997,[],[https://image-cdn.beforward.jp/large/202604/1...,Truck


In [102]:
# Example usage with identity columns
identity_report = missing_report(df, identity_columns)
print(identity_report)

# Rows where any of these columns are null
mask = df[[c for c in identity_columns if c in df.columns]].isnull().any(axis=1)
problem_rows = df.loc[mask, [c for c in identity_columns if c in df.columns]]
problem_rows

         missing_count  missing_pct
source               0          0.0
ref_no               0          0.0
car_url              0          0.0


,source,ref_no,car_url


In [103]:
vehicle_report = missing_report(df, vehicle_columns)
print(vehicle_report)
mask = df[[c for c in vehicle_columns if c in df.columns]].isnull().any(axis=1)
problem_rows = df.loc[mask, [c for c in vehicle_columns if c in df.columns]]
problem_rows.head()

                  missing_count  missing_pct
make                          0          0.0
model                         0          0.0
model_code                    0          0.0
grade                         0          0.0
manufacture_year              0          0.0
mileage                       0          0.0
engine_capacity               0          0.0
fuel_type                     0          0.0
transmission                  0          0.0
body_type                     0          0.0
drive_type                    0          0.0
seats                         0          0.0
doors                         0          0.0
steering                      0          0.0
exterior_color                0          0.0


,make,model,model_code,grade,manufacture_year,mileage,engine_capacity,fuel_type,transmission,body_type,drive_type,seats,doors,steering,exterior_color


In [104]:
import pandas as pd
import re
from datetime import datetime

# ────────────────────────────────────────────────────────────────
# Helper functions
# ────────────────────────────────────────────────────────────────

def extract_int(val):
    """Pull the first integer from a messy mileage / capacity string."""
    if pd.isna(val) or val == '':
        return None
    # Remove "km", commas, and keep only digits
    t = str(val).lower().split("km")[0].replace(",", "")
    digits = re.sub(r"[^\d]", "", t)
    return int(digits) if digits else None

def extract_year(val):
    """Extract a 4-digit year from the beginning or anywhere."""
    if pd.isna(val) or val == '':
        return None
    s = str(val).strip()
    m = re.match(r"(\d{4})", s)
    if m:
        y = int(m.group(1))
        return y if 1900 <= y <= 2100 else None
    # If year is not at start, try to find any 4-digit block
    m = re.search(r"\b(\d{4})\b", s)
    if m:
        y = int(m.group(1))
        return y if 1900 <= y <= 2100 else None
    return None

def extract_engine_cc(val):
    """Extract engine capacity in cc as an integer."""
    if pd.isna(val) or val == '':
        return None
    m = re.search(r"(\d[\d,]*)", str(val))
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def safe_int(val):
    """Convert a value to int, returning None if impossible."""
    if pd.isna(val) or str(val).strip() == '':
        return None
    try:
        return int(float(str(val).strip()))
    except (ValueError, TypeError):
        return None

def safe_numeric(val):
    """Convert to float, return None on failure."""
    if pd.isna(val) or str(val).strip() == '':
        return None
    try:
        return float(str(val).strip().replace(",", ""))
    except (ValueError, TypeError):
        return None

# ────────────────────────────────────────────────────────────────
# Main conversion function
# ────────────────────────────────────────────────────────────────

def convert_car_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast columns to their proper data types.
    Creates new columns if needed (mileage_km, engine_capacity_cc) and
    cleans existing ones.
    """
    # 1. Years
    if 'manufacture_year' in df.columns:
        df['manufacture_year'] = df['manufacture_year'].apply(extract_year)
    if 'registration_year' in df.columns:
        df['registration_year'] = df['registration_year'].apply(extract_year)

    # 2. Numeric mileage (add new column, keep original string)
    if 'mileage' in df.columns:
        df['mileage_km'] = df['mileage'].apply(extract_int)

    # 3. Engine capacity (add new column)
    if 'engine_capacity' in df.columns:
        df['engine_capacity_cc'] = df['engine_capacity'].apply(extract_engine_cc)

    # 4. Seats & doors
    for col in ['seats', 'doors']:
        if col in df.columns:
            df[col] = df[col].apply(safe_int).astype('Int64')  # nullable integer

    # 5. Price columns (USD amounts)
    price_cols = ['vehicle_price', 'total_price', 'original_price',
                  'freight_amount', 'inspection_amount', 'insurance_amount']
    for col in price_cols:
        if col in df.columns:
            df[col] = df[col].apply(safe_numeric).astype('float64')

    # 6. Scraped timestamp
    if 'scraped_at' in df.columns:
        df['scraped_at'] = pd.to_datetime(df['scraped_at'], errors='coerce')

    # 7. Ensure strings are clean (strip) for text columns
    text_cols = ['source', 'car_url', 'ref_no', 'title', 'model_code', 'grade',
                 'body_type', 'transmission', 'fuel_type', 'drive_type',
                 'steering', 'exterior_color', 'chassis_no', 'delivery_port',
                 'location', 'make', 'model', 'model_name',
                 'discount_rate']
    for col in text_cols:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip().replace({'nan': '', 'None': ''})

    return df

# ────────────────────────────────────────────────────────────────
# Apply conversion
# ────────────────────────────────────────────────────────────────

df = convert_car_dtypes(df)

# Confirm the new dtypes
print(df.dtypes)
df.head(3)

source                         str
car_url                        str
title                          str
model_code                     str
ref_no                         str
registration_year          float64
manufacture_year           float64
grade                          str
transmission                   str
mileage                        str
engine_capacity                str
fuel_type                      str
seats                        Int64
doors                        Int64
steering                       str
drive_type                     str
exterior_color                 str
chassis_no                     str
currency                       str
vehicle_price              float64
total_price                float64
discount_rate                  str
delivery_port                  str
location                       str
body_type                      str
freight_amount             float64
inspection_amount          float64
insurance_amount           float64
vehicle_price_breakd

,source,car_url,title,model_code,ref_no,registration_year,manufacture_year,grade,transmission,mileage,...,insurance_amount,vehicle_price_breakdown,make,model,model_name,features,image_urls,body_type_from_title,mileage_km,engine_capacity_cc
0,beforward,https://www.beforward.jp/toyota/voxy/cc101493/...,2009 TOYOTA VOXY ZS KIRAMEKI,DBA-ZRR75W,CC101493,2009.0,2009.0,ZS KIRAMEKI,Automatic,"178,375 km",...,0.0,0,Toyota,VOXY ZS KIRAMEKI,2009,"[CD Player, Power Steering, Power Window, A/C,...",[https://image-cdn.beforward.jp/large/202604/1...,,178375.0,1980
1,beforward,https://www.beforward.jp/toyota/hiace-van/cc88...,2020 TOYOTA HIACE VAN DX,QDF-GDH206V,CC889261,2020.0,NaN,DX,Automatic,"74,375 km",...,0.0,0,Toyota,HIACE DX,2020,"[Power Steering, Power Window, A/C, ABS, Airba...",[https://image-cdn.beforward.jp/large/202604/1...,Van,74375.0,2750
2,beforward,https://www.beforward.jp/isuzu/elf-truck/cd066...,1997 ISUZU ELF TRUCK,KC-NKR66ED,CD066063,1997.0,NaN,,Manual,"116,333 km",...,0.0,0,Isuzu,ELF,1997,[],[https://image-cdn.beforward.jp/large/202604/1...,Truck,116333.0,4330


In [105]:
print(missing_report(df, df.columns))

                         missing_count  missing_pct
source                               0          0.0
car_url                              0          0.0
title                                0          0.0
model_code                           0          0.0
ref_no                               0          0.0
registration_year                   50         50.0
manufacture_year                    43         43.0
grade                                0          0.0
transmission                         0          0.0
mileage                              0          0.0
engine_capacity                      0          0.0
fuel_type                            0          0.0
seats                               27         27.0
doors                                0          0.0
steering                             0          0.0
drive_type                           0          0.0
exterior_color                       0          0.0
chassis_no                           0          0.0
currency    